# Data Description
- Author: 李芸榛（JenniferjjLee）
- Date first created: 20261108
- Last updated: 20261108（輸入改為 v7 交易檔，修正 `捷運科技大樓站(台北教育大學)` 的截斷站名後重新產生主檔；同日改用 109 站的新版 `stations_in_field` 再重新產生，臺大人文館改用正式 StationID，並納入羅斯福路三段283巷；同日主檔改為 Parquet，並刪除範例檔）
- Background: 研究範圍內 109 站（`stations_in_field.csv` 2026-11-08 版，含 2025 年才啟用的臺大人文館 `500119097` 與羅斯福路三段283巷 `500101265`）每小時的 YouBike 借出量、還入量與淨流量，由 v7 台大周邊交易資料（2024、2025）彙總而成，並依 `trip_group` 拆分旅次類型。範圍外站點只收到和台大有往來的旅次，流量不完整，所以不列入。
- Unit of observation: 一個站點的一個小時（`station_id` × `datetime`，唯一且完整）。
- Sample period: 2024-01-01 00 時～2025-12-31 23 時，共 17,544 小時；109 站 × 17,544 小時 = 1,912,296 列，沒有借還的小時填 0。
- Known issues:
  - **時間只精確到小時**：原始交易的借還時間都是整點，所以只能做到小時層級。
  - **淨流量不含調度**：只反映騎乘，看不到調度車補車或載走，不能直接當作站上車數的變化。
  - **`in_service = 0` 不是沒人借**：表示該站當時還沒啟用（75,576 列）。啟用日 = 站位快照的 `first_seen` 和該站第一筆交易日期取較早者；臺大人文館從 2025-02-04 起，羅斯福路三段283巷從 2025-10-10 起。`臺大共同教室北側` 從 2024-01-02 起標為啟用，但原始資料到 2024-06 才出現這個站名，之前雖然啟用、借還都是 0。（v6 版的 `捷運科技大樓站(台北教育大學)` 也有同樣現象，原因是截斷站名沒有對應到，v7 已修正，該站從 2024-01-01 起就有交易。）
  - **啟用中仍常為 0**：啟用中的站點小時裡，19.95% 完全沒有借還（多在深夜）。
  - **還入少 75 筆**：還車時間超出 2025-12-31 23 時的 75 筆旅次（2026-01-01 00–07 時的跨年還車，以及 1 筆 2034-08-26 的錯誤紀錄）放不進時間格，沒有計入。
  - **無效借還（d）只在 2025-08 以後**：資料來源在 2025-08 以前就沒有 5 分鐘內的同站還車紀錄，`rent_invalid`／`return_invalid` 不能做跨期比較。
  - **2025 年新站**：臺大人文館與羅斯福路三段283巷只在 2025 年啟用，跨年比較需要控制站點數或只用 `in_service = 1` 的列。（前版臺大人文館暫用 `-2`、283巷算範圍外，本版已改正。）
- Definition for each variable:
  - `station_id`：StationID（9 碼）。
  - `station_name`：站名，含 `YouBike2.0_` 前綴。
  - `datetime`：該小時的開始時間 `YYYY-MM-DD HH:00:00`；例如 08:00:00 代表 08:00–08:59。
  - `in_service`：該站在該小時是否已啟用（1／0）。
  - `rent_total`：該站該小時的借出筆數（以借車時間計）。
    - `rent_to_outside`：借出後還到範圍外（trip_group b，台大借外還）。
    - `rent_to_field`：借出後還到範圍內（c，台大借台大還）。
    - `rent_invalid`：無效借還（d，同站借還且時長 ≤ 3 分鐘）。
  - `return_total`：該站該小時的還入筆數（以還車時間計）。
    - `return_from_outside`：從範圍外借來（a，外借台大還）。
    - `return_from_field`：從範圍內借來（c）。
    - `return_invalid`：無效借還（d）。
  - `net_flow`：`return_total` − `rent_total`，正值 = 該小時騎乘使車輛增加。
- Data path:
  - `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet`（1,912,296 列 × 13 欄，Parquet／zstd，約 9.9 MB，可以 commit；`station_id`、`station_name`、`datetime` 為字串，其餘為 int64；先前的 CSV 版已刪除）
- Source data: `data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet`、`YouBike_traffic_2025_v7.parquet`（說明見 [YouBike_traffic_doc.ipynb](../youbike_traffic/YouBike_traffic_doc.ipynb)），以及 `data/processed/stations_in_field.csv`。
- Processing code: [data/code/youbike_station_hourly_flow.ipynb](../../code/youbike_station_hourly_flow.ipynb)
- Role: `analysis`。
- Verification status: 2026-11-08 在全新的 in-process IPython shell 由上而下執行本 notebook（本機沒有 Jupyter）：SHA-256、欄位、列數、站點 × 小時唯一且完整、時間範圍、計數非負、各類加總與淨流量、未啟用時為 0、各欄加總等於產生程式的紀錄、範例檔內容，全部通過（見 Assertions）；同日改用 v7 產生的主檔後重跑，全部通過；改用 109 站清單重新產生主檔後再重跑，全部通過；主檔改為 Parquet 後再重跑，全部通過；刪除範例檔並移除相關檢查後再重跑，全部通過。
- SHA-256:
  - 主檔：`b2cb22824c899f72afe7eda188afe6131eb7fa419f703b8728f52b26e2e332db`
- Downstream analysis input: `data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet`（與 Data path 相同）


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


import numpy as np
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Change for Your Own Data

下方設定綁定本文件描述的主檔；路徑不存在或 SHA-256 不符就停止，不會改讀其他版本。`EXPECTED_TOTALS` 是產生程式記錄的各欄加總，用來核對檔案內容。

In [2]:
# Exact files documented here (produced by data/code/youbike_station_hourly_flow.ipynb).
DIR = "data/processed/youbike_station_hourly_flow"
MAIN = {"path": f"{DIR}/youbike_station_hourly_flow_2024_2025.parquet", "rows": 1_912_296,
        "sha256": "b2cb22824c899f72afe7eda188afe6131eb7fa419f703b8728f52b26e2e332db"}
EXPECTED_COLUMNS = ["station_id", "station_name", "datetime", "in_service",
                    "rent_total", "rent_to_outside", "rent_to_field", "rent_invalid",
                    "return_total", "return_from_outside", "return_from_field", "return_invalid", "net_flow"]
COUNT_COLUMNS = EXPECTED_COLUMNS[4:]
RENT_PARTS = ["rent_to_outside", "rent_to_field", "rent_invalid"]
RETURN_PARTS = ["return_from_outside", "return_from_field", "return_invalid"]
N_STATIONS, N_HOURS = 109, 17_544
# Column totals recorded by the processing notebook (trip counts from the v7 files).
EXPECTED_TOTALS = {"rent_total": 14_110_392, "rent_to_outside": 4_261_858, "rent_to_field": 9_845_805, "rent_invalid": 2_729,
                   "return_total": 13_768_960, "return_from_outside": 3_920_467, "return_from_field": 9_845_764,
                   "return_invalid": 2_729, "net_flow": -341_432}
READ = {"dtype": {"station_id": str, "station_name": str, "datetime": str}, "encoding": "utf-8-sig"}

## Summary

先核對 SHA-256 再讀入。`_dt` 是為了核對而暫時解析的時間欄，不寫回檔案。

In [3]:
path = project_file(MAIN["path"], "processed")
MAIN["fingerprint"] = verify_input_sha256(path, MAIN["sha256"])
df = pd.read_parquet(path)
print({"data_path": MAIN["path"], "rows": len(df), "schema": {c: str(t) for c, t in df.dtypes.items()},
       "sha256": MAIN["fingerprint"]})
df["_dt"] = pd.to_datetime(df["datetime"], format="%Y-%m-%d %H:%M:%S")   # for checks only

{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet', 'rows': 1912296, 'schema': {'station_id': 'object', 'station_name': 'object', 'datetime': 'object', 'in_service': 'int64', 'rent_total': 'int64', 'rent_to_outside': 'int64', 'rent_to_field': 'int64', 'rent_invalid': 'int64', 'return_total': 'int64', 'return_from_outside': 'int64', 'return_from_field': 'int64', 'return_invalid': 'int64', 'net_flow': 'int64'}, 'sha256': 'b2cb22824c899f72afe7eda188afe6131eb7fa419f703b8728f52b26e2e332db'}


### Sample

In [4]:
print("主檔依檔案順序的前 10 列（捷運科技大樓站，2024-01-01 00–09 時）")
display(df[EXPECTED_COLUMNS].head(10))

主檔依檔案順序的前 10 列（捷運科技大樓站，2024-01-01 00–09 時）


,station_id,station_name,datetime,in_service,rent_total,rent_to_outside,rent_to_field,rent_invalid,return_total,return_from_outside,return_from_field,return_invalid,net_flow
0,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 00:00:00,1,13,4,9,0,7,3,4,0,-6
1,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 01:00:00,1,14,5,9,0,11,10,1,0,-3
2,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 02:00:00,1,3,0,3,0,4,0,4,0,1
3,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 03:00:00,1,13,6,7,0,9,6,3,0,-4
4,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 04:00:00,1,7,0,7,0,6,2,4,0,-1
5,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 05:00:00,1,4,1,3,0,4,1,3,0,0
6,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 06:00:00,1,3,1,2,0,3,3,0,0,0
7,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 07:00:00,1,4,2,2,0,8,7,1,0,4
8,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 08:00:00,1,8,3,5,0,17,9,8,0,9
9,500101001,YouBike2.0_捷運科技大樓站,2024-01-01 09:00:00,1,6,4,2,0,9,7,2,0,3


### Summary Stats

In [5]:
def fmt_int(x): return f"{int(round(x)):,}"
def fmt2(x): return f"{x:,.2f}"


def numeric_summary(frame):
    return pd.DataFrame({c: {"N": fmt_int(frame[c].notna().sum()), "缺值 N": fmt_int(frame[c].isna().sum()),
                             "平均數": fmt2(frame[c].mean()), "標準差": fmt2(frame[c].std()), "最小值": fmt_int(frame[c].min()),
                             "P25": fmt2(frame[c].quantile(.25)), "中位數": fmt2(frame[c].median()),
                             "P75": fmt2(frame[c].quantile(.75)), "最大值": fmt_int(frame[c].max()),
                             "合計": fmt_int(frame[c].sum())} for c in COUNT_COLUMNS}).T


print(f"表 1　每站每小時流量（全部 {len(df):,} 列）")
display(numeric_summary(df))
active = df[df["in_service"] == 1]
print(f"表 2　每站每小時流量（只含 in_service = 1 的 {len(active):,} 列）")
display(numeric_summary(active))

print("表 3　類別欄位")
display(pd.DataFrame({
    "station_id": {"N": fmt_int(df["station_id"].notna().sum()), "不同值數": df["station_id"].nunique(),
                   "每站列數": f"{df.groupby('station_id').size().min():,}～{df.groupby('station_id').size().max():,}"},
    "datetime": {"N": fmt_int(df["datetime"].notna().sum()), "不同值數": f"{df['datetime'].nunique():,}",
                 "每小時列數": f"{df.groupby('datetime').size().min()}～{df.groupby('datetime').size().max()}"},
    "in_service": {"N": fmt_int(len(df)), "不同值數": df["in_service"].nunique(),
                   "每小時列數": f"1：{(df['in_service'] == 1).sum():,}（{(df['in_service'] == 1).mean():.2%}）；0：{(df['in_service'] == 0).sum():,}"},
}).T)

zero_share = ((active["rent_total"] == 0) & (active["return_total"] == 0)).mean()
print(f"啟用中的站點小時裡，完全沒有借還的占 {zero_share:.2%}。")
by_station = df.groupby(["station_id", "station_name"])[["rent_total", "return_total", "net_flow"]].sum()
by_station["hours_in_service"] = df.groupby(["station_id", "station_name"])["in_service"].sum()
print("表 4　借出量前 10 站（兩年合計）")
display(by_station.sort_values("rent_total", ascending=False).head(10).map(fmt_int))
first_active = df[df["in_service"] == 1].groupby(["station_id", "station_name"])["datetime"].min()
print("表 5　啟用時間晚於 2024-01-01 00 時的站")
display(first_active[first_active > "2024-01-01 00:00:00"].rename("in_service 起始").to_frame())

表 1　每站每小時流量（全部 1,912,296 列）


,N,缺值 N,平均數,標準差,最小值,P25,中位數,P75,最大值,合計
rent_total,"1,912,296",0,7.38,13.88,0,0.00,3.00,9.00,283,"14,110,392"
rent_to_outside,"1,912,296",0,2.23,5.17,0,0.00,0.00,2.00,147,"4,261,858"
rent_to_field,"1,912,296",0,5.15,10.16,0,0.00,2.00,6.00,271,"9,845,805"
rent_invalid,"1,912,296",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,729"
return_total,"1,912,296",0,7.20,14.73,0,0.00,2.00,8.00,380,"13,768,960"
return_from_outside,"1,912,296",0,2.05,4.92,0,0.00,0.00,2.00,142,"3,920,467"
return_from_field,"1,912,296",0,5.15,11.05,0,0.00,2.00,6.00,323,"9,845,764"
return_invalid,"1,912,296",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,729"
net_flow,"1,912,296",0,-0.18,7.99,-204,-1.00,0.00,1.00,249,"-341,432"


表 2　每站每小時流量（只含 in_service = 1 的 1,836,720 列）


,N,缺值 N,平均數,標準差,最小值,P25,中位數,P75,最大值,合計
rent_total,"1,836,720",0,7.68,14.08,0,0.00,3.00,9.00,283,"14,110,392"
rent_to_outside,"1,836,720",0,2.32,5.25,0,0.00,1.00,3.00,147,"4,261,858"
rent_to_field,"1,836,720",0,5.36,10.31,0,0.00,2.00,6.00,271,"9,845,805"
rent_invalid,"1,836,720",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,729"
return_total,"1,836,720",0,7.50,14.96,0,0.00,3.00,9.00,380,"13,768,960"
return_from_outside,"1,836,720",0,2.13,5.01,0,0.00,1.00,2.00,142,"3,920,467"
return_from_field,"1,836,720",0,5.36,11.22,0,0.00,2.00,6.00,323,"9,845,764"
return_invalid,"1,836,720",0,0.00,0.04,0,0.00,0.00,0.00,5,"2,729"
net_flow,"1,836,720",0,-0.19,8.16,-204,-1.00,0.00,1.00,249,"-341,432"


表 3　類別欄位


,N,不同值數,每站列數,每小時列數
station_id,"1,912,296",109,"17,544～17,544",NaN
datetime,"1,912,296","17,544",NaN,109～109
in_service,"1,912,296",2,NaN,"1：1,836,720（96.05%）；0：75,576"


啟用中的站點小時裡，完全沒有借還的占 19.95%。


表 4　借出量前 10 站（兩年合計）


,,rent_total,return_total,net_flow,hours_in_service
station_id,station_name,,,,
500101022,YouBike2.0_捷運公館站(2號出口),"1,059,785","1,125,624","65,839","17,544"
500101181,YouBike2.0_捷運公館站(3號出口),"661,328","700,731","39,403","17,544"
500101001,YouBike2.0_捷運科技大樓站,"650,484","629,552","-20,932","17,544"
500119075,YouBike2.0_臺大第一活動中心西南側,"396,275","426,877","30,602","17,544"
500119090,YouBike2.0_臺大新體育館東南側,"307,849","314,664","6,815","17,544"
500119050,YouBike2.0_臺大小福樓東側,"297,480","316,283","18,803","17,544"
500101032,YouBike2.0_臺大男一舍前,"283,023","272,953","-10,070","17,544"
500119067,YouBike2.0_臺大社科院西側,"252,191","251,646",-545,"17,544"
500119049,YouBike2.0_臺大女九舍西南側,"236,957","246,904","9,947","17,544"


表 5　啟用時間晚於 2024-01-01 00 時的站


,,in_service 起始
station_id,station_name,
500101234,YouBike2.0_和平東路二段90巷,2024-03-19 00:00:00
500101240,YouBike2.0_和平東路二段76巷,2024-09-20 00:00:00
500101250,YouBike2.0_國立臺北教育大學創意館,2024-12-13 00:00:00
500101265,YouBike2.0_羅斯福路三段283巷,2025-10-10 00:00:00
500106135,YouBike2.0_羅斯福路四段108巷口,2024-10-15 00:00:00
500119059,YouBike2.0_臺大共同教室北側,2024-01-02 00:00:00
500119092,YouBike2.0_臺大管理學院一館,2024-08-13 00:00:00
500119093,YouBike2.0_臺大禮賢樓西側,2024-08-16 00:00:00
500119094,YouBike2.0_臺大大一女餐廳廣場,2024-08-13 00:00:00


### Assertions

檢查由產生規則支持的性質：欄位、列數、站點 × 小時唯一且完整、時間範圍與整點、站名一致、計數非負、各類加總等於總量、淨流量定義、未啟用時借還為 0 且啟用後不再變回 0、欄位加總等於產生程式的紀錄、無效借還只出現在 2025-08 以後。

In [6]:
assert list(df.columns[:13]) == EXPECTED_COLUMNS, "schema differs"
assert len(df) == MAIN["rows"] == N_STATIONS * N_HOURS
assert not df.duplicated(["station_id", "datetime"]).any(), "station × hour is not unique"
assert df["station_id"].nunique() == N_STATIONS and df["datetime"].nunique() == N_HOURS
assert df["_dt"].min() == pd.Timestamp("2024-01-01 00:00") and df["_dt"].max() == pd.Timestamp("2025-12-31 23:00")
assert (df["_dt"].dt.minute == 0).all() and (df["_dt"].dt.second == 0).all()
assert df.groupby("station_id")["station_name"].nunique().max() == 1
assert df["station_id"].str.fullmatch(r"\d{9}").all()
assert df["in_service"].isin([0, 1]).all()
assert df[COUNT_COLUMNS].notna().all().all()
assert (df[COUNT_COLUMNS[:-1]] >= 0).all().all(), "negative count"
assert (df["rent_total"] == df[RENT_PARTS].sum(axis=1)).all()
assert (df["return_total"] == df[RETURN_PARTS].sum(axis=1)).all()
assert (df["net_flow"] == df["return_total"] - df["rent_total"]).all()
assert (df.loc[df["in_service"] == 0, ["rent_total", "return_total"]] == 0).all().all(), "flow before in_service"
# Once a station is in service it stays in service.
ordered = df.sort_values(["station_id", "_dt"])
assert (ordered.groupby("station_id")["in_service"].diff().fillna(0) >= 0).all(), "in_service switches back to 0"
assert df[COUNT_COLUMNS].sum().to_dict() == EXPECTED_TOTALS, "totals differ from the processing record"
assert (df.loc[df["rent_invalid"] + df["return_invalid"] > 0, "datetime"] >= "2025-08-01").all(), "invalid flow before 2025-08"
assert sha256_file(path) == MAIN["fingerprint"]
print({"data_path": MAIN["path"], "rows": len(df), "columns": 13, "sha256": MAIN["fingerprint"], "status": "configured checks passed"})

{'data_path': 'data/processed/youbike_station_hourly_flow/youbike_station_hourly_flow_2024_2025.parquet', 'rows': 1912296, 'columns': 13, 'sha256': 'b2cb22824c899f72afe7eda188afe6131eb7fa419f703b8728f52b26e2e332db', 'status': 'configured checks passed'}


### 本次文件核對的結論與限制

- 主檔與產生程式的紀錄一致，所有 assertion 通過。
- 每站每小時的借出量平均 7.38 筆（中位數 3），還入量平均 7.20 筆；單一站點單一小時最多借出 283 筆、還入 380 筆，淨流量介於 −204 到 +249。分佈高度右偏，大量站點小時接近 0。
- 兩年合計：借出 14,110,392 筆（台大借外還 4,261,858、台大借台大還 9,845,805、無效 2,729），還入 13,768,960 筆（外借台大還 3,920,467、台大借台大還 9,845,764、無效 2,729），淨流量 −341,432。
- 改用 109 站清單後：新增羅斯福路三段283巷（借出 7,155、還入 6,833）；其他 108 站每小時的借出、還入、淨流量與 `in_service` 都不變，只有和 283巷 往來的旅次改算範圍內，細項拆分改變。
- 與 v6 版相比（改用 109 站清單前），只有 `500101166`（捷運科技大樓站(台北教育大學)）的借還量改變（2024 年借出 24 → 88,116、還入 27 → 83,182）；其他 107 站每小時的借出、還入與淨流量都相同，只是範圍內／外的細項拆分改變。
- 借出量最多的是捷運公館站(2號出口)（1,059,785 筆，淨流量 +65,839）、公館站(3號出口) 和捷運科技大樓站。
- 本文件只描述與核對，沒有修改任何資料。
